# 05 — Incident ranking and locked holdout

This notebook ranks the selected development alerts without rescoring them.
Set `RUN_HOLDOUT=1` only after the selected configuration has been
reviewed and frozen; no tuning is allowed after that run.

## 1. Setup

In [ ]:
import hashlib
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT")
    or os.getenv("ANOMALY_DRIVE_ROOT")
    or (
        "/content/drive/MyDrive/anomaly_detection"
        if IN_COLAB else Path.home() / "anomaly_detection_data"
    )
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

SECTOR = os.getenv("ANOMALY_SECTOR", "telecom")
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_10_1_run1",
    "petrobras_3w": "petrobras_3w_core_v0_10_1_run1",
}
if SECTOR not in CANONICAL_RUN_IDS:
    raise ValueError(f"Choose one of {list(CANONICAL_RUN_IDS)}")

import tempfile

import joblib
import matplotlib.pyplot as plt

from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json
from evaluation_core import evaluate_alerts
from simple_model_core import (
    MODEL_CORE_VERSION, alerts_from_score_file, materialize_features,
    materialize_wide_partition,
    partition_exposure, score_feature_file, score_percentiles,
)

EDA_VERSION = "1.3.0"
EVALUATION_VERSION = "1.3.0"
MODEL_VERSION = FINAL_VERSION = "1.3.0"
CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
EDA_RUN_ID = os.getenv("EDA_RUN_ID", f"{SECTOR}_eda_v1_3_run1")
EVALUATION_RUN_ID = os.getenv("EVALUATION_RUN_ID", f"{SECTOR}_evaluation_v1_3_run1")
MODEL_RUN_ID = os.getenv("MODEL_RUN_ID", f"{SECTOR}_models_v1_3_run1")
FINAL_RUN_ID = os.getenv("FINAL_RUN_ID", f"{SECTOR}_incidents_v1_3_run1")
RUN_HOLDOUT = os.getenv("RUN_HOLDOUT", "0") == "1"

RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / CANONICAL_RUN_ID
CORE_ROOT, SPLIT_ROOT = RUN_ROOT / "SPEC-CORE", RUN_ROOT / "SPLITS"
EVALUATION_ROOT = DATA_ROOT / "outputs" / "evaluation" / f"v{EVALUATION_VERSION}" / SECTOR / EVALUATION_RUN_ID
MODEL_ROOT = DATA_ROOT / "outputs" / "models" / f"v{MODEL_VERSION}" / SECTOR / MODEL_RUN_ID
PARTITION = "holdout" if RUN_HOLDOUT else "development"
FINAL_ROOT = (
    DATA_ROOT / "outputs" / "final" / f"v{FINAL_VERSION}"
    / SECTOR / FINAL_RUN_ID / PARTITION
)

bundle = joblib.load(MODEL_ROOT / "selected_model.joblib")
configuration = read_json(MODEL_ROOT / "selected_configuration.json")
model_fault_type_comparison = pd.read_csv(
    MODEL_ROOT / "model_comparison_by_fault_type.csv"
)
manifest = read_json(CORE_ROOT / "manifest.json")
policy = read_json(EVALUATION_ROOT / "evaluation_policy.json")
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
entity_groups_path = SPLIT_ROOT / "entity_groups.parquet"
entity_groups = pd.read_parquet(entity_groups_path) if entity_groups_path.is_file() else pd.DataFrame()
decisions = configuration["feature_settings"]
if manifest["sector"] != SECTOR:
    raise ValueError("Canonical sector does not match the requested sector")
if decisions["eda_version"] != EDA_VERSION:
    raise ValueError("Model features are not from the declared EDA version")
if policy["evaluation_version"] != EVALUATION_VERSION:
    raise ValueError("Evaluation policy is not from the declared version")
if configuration["canonical_fingerprint"] != manifest["fingerprint"]:
    raise ValueError("The selected model belongs to a different canonical run")
if configuration["model_core_version"] != MODEL_CORE_VERSION:
    raise ValueError("Re-run Notebook 04 with the current simple_model_core.py")
canonical_build_ts = pd.Timestamp(
    (CORE_ROOT / "manifest.json").stat().st_mtime, unit="s", tz="UTC"
)
configuration_path = MODEL_ROOT / "selected_configuration.json"
configuration_fingerprint = hashlib.sha256(
    configuration_path.read_bytes()
).hexdigest()
prediction_path = MODEL_ROOT / "holdout_prediction.json"
receipt_path = EVALUATION_ROOT / "holdout_sealed" / "holdout_receipt.json"
if FINAL_ROOT.exists():
    raise FileExistsError(f"Final output already exists: {FINAL_ROOT}")
holdout_prediction = None
if RUN_HOLDOUT:
    if receipt_path.exists():
        raise RuntimeError("Holdout has already been opened for this evaluation run")
    if not prediction_path.is_file():
        raise FileNotFoundError(
            "Create holdout_prediction.json from the template before opening holdout"
        )
    holdout_prediction = read_json(prediction_path)
    predictions = holdout_prediction.get("predictions", {})
    required_prediction_metrics = {
        policy["primary_selection_metric"],
        f"false_alerts_per_{policy['primary_exposure_unit']}",
    }
    if set(predictions) != required_prediction_metrics:
        raise ValueError("Holdout prediction must declare every primary metric")
    for metric_name, values in predictions.items():
        required = {"expected", "confirmation_min", "confirmation_max"}
        if set(values) != required or any(values[name] is None for name in required):
            raise ValueError(f"Incomplete holdout prediction for {metric_name}")
        if not values["confirmation_min"] <= values["expected"] <= values["confirmation_max"]:
            raise ValueError(f"Invalid confirmation range for {metric_name}")
    write_json(receipt_path, {
        "opened_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "canonical_fingerprint": manifest["fingerprint"],
        "configuration_fingerprint": configuration_fingerprint,
    })

display(pd.Series({
    "sector": SECTOR, "selected_model": configuration["model_id"],
    "selection_status": configuration["selection_status"],
    "partition": PARTITION, "final_output": str(FINAL_ROOT),
    "canonical_run_id": CANONICAL_RUN_ID,
    "canonical_build_ts": canonical_build_ts,
}, name="value").to_frame())

## 2. Score one frozen partition

In [ ]:
work = tempfile.TemporaryDirectory(prefix=f"{SECTOR}-final-")
WORK_ROOT = Path(work.name)
decisions = configuration["feature_settings"]
lookback_seconds = max(decisions["rolling_windows_seconds"].values())

def score_partition(partition):
    wide = WORK_ROOT / f"{partition}_wide.parquet"
    bounds = materialize_wide_partition(
        CORE_ROOT, SPLIT_ROOT, partition, catalogue, wide,
        lookback_seconds=lookback_seconds,
        memory_limit=os.getenv("ANOMALY_DUCKDB_MEMORY_LIMIT", "3GB"),
        threads=int(os.getenv("ANOMALY_DUCKDB_THREADS", "2")),
    )
    features = WORK_ROOT / f"{partition}_features.parquet"
    materialize_features(
        wide, catalogue, decisions, features,
        score_start=bounds["score_start"], score_end=bounds["score_end"],
    )
    scores = WORK_ROOT / f"{partition}_scores.parquet"
    score_feature_file(bundle, features, scores)
    alerts = alerts_from_score_file(
        scores, configuration["model_id"], configuration["threshold"],
        min_consecutive=configuration["persistence_observations"],
        recovery_consecutive=configuration["recovery_observations"],
    )
    if not alerts.empty:
        alerts["score_percentile"] = score_percentiles(
            bundle, configuration["model_id"], alerts["peak_score"]
        )
    else:
        alerts["score_percentile"] = pd.Series(dtype=float)
    if configuration["threshold"] == 0:
        raise ValueError("Exceedance ratio requires a non-zero threshold")
    alerts["exceedance_ratio"] = (
        alerts["peak_score"] / configuration["threshold"]
    )
    return scores, alerts

## 3. Convert entity alerts into incidents

In [ ]:
def rank_incidents(alerts, groups, grouping_window_seconds):
    if alerts.empty:
        return pd.DataFrame(columns=[
            "incident_id", "start_ts", "end_ts", "scope_type", "scope_id",
            "alert_count", "alert_ids", "affected_entity_count",
            "affected_entities", "peak_exceedance_ratio",
            "peak_score_percentile",
            "anomalous_metric_count", "duration_seconds",
            "leading_metrics", "rank",
        ])

    alerts = alerts.sort_values("alert_start").reset_index(drop=True).copy()
    memberships = {}
    group_priority = []
    if not groups.empty:
        group_priority = (
            groups.groupby("group_type")["group_id"].nunique()
            .rename("groups").reset_index()
            .sort_values(["groups", "group_type"], ascending=[False, True])
            ["group_type"].tolist()
        )
        for entity_id, frame in groups.groupby("entity_id"):
            memberships[str(entity_id)] = set(zip(frame["group_type"], frame["group_id"].astype(str)))

    parent = list(range(len(alerts)))
    def find(index):
        while parent[index] != index:
            parent[index] = parent[parent[index]]
            index = parent[index]
        return index
    def union(left, right):
        left, right = find(left), find(right)
        if left != right:
            parent[right] = left

    window = pd.Timedelta(seconds=grouping_window_seconds)
    active = []
    for right in range(len(alerts)):
        cutoff = alerts.loc[right, "alert_start"] - window
        active = [left for left in active if alerts.loc[left, "alert_end"] >= cutoff]
        for left in active:
            left_groups = memberships.get(str(alerts.loc[left, "entity_id"]), set())
            right_groups = memberships.get(str(alerts.loc[right, "entity_id"]), set())
            if left_groups & right_groups:
                union(left, right)
        active.append(right)

    components = {}
    for index in range(len(alerts)):
        components.setdefault(find(index), []).append(index)

    rows = []
    for indices in components.values():
        group = alerts.loc[indices]
        entities = sorted(group["entity_id"].astype(str).unique())
        common = None
        for entity_id in entities:
            common = memberships.get(entity_id, set()) if common is None else common & memberships.get(entity_id, set())
        common = common or set()
        if len(entities) == 1:
            scope = ("entity", entities[0])
        else:
            scope = next(
                ((kind, value) for kind in group_priority
                 for pair_kind, value in sorted(common) if pair_kind == kind),
                ("multi_entity", "unresolved"),
            )
        metrics = sorted({
            feature.split("__", 1)[0]
            for feature in group["leading_feature"].dropna().astype(str)
        })
        start, end = group["alert_start"].min(), group["alert_end"].max()
        rows.append({
            "start_ts": start, "end_ts": end,
            "scope_type": scope[0], "scope_id": scope[1],
            "alert_count": len(group),
            "alert_ids": ", ".join(group["alert_id"].astype(str)),
            "affected_entity_count": len(entities),
            "affected_entities": ", ".join(entities),
            "peak_exceedance_ratio": group["exceedance_ratio"].max(),
            "peak_score_percentile": group["score_percentile"].max(),
            "anomalous_metric_count": len(metrics),
            "duration_seconds": (end - start).total_seconds(),
            "leading_metrics": ", ".join(metrics[:5]),
        })
    incidents = pd.DataFrame(rows).sort_values(
        ["affected_entity_count", "anomalous_metric_count",
         "peak_exceedance_ratio", "duration_seconds", "start_ts"],
        ascending=[False, False, False, False, True],
    ).reset_index(drop=True)
    incidents.insert(0, "incident_id", [f"I-{number:06d}" for number in range(1, len(incidents) + 1)])
    incidents["rank"] = np.arange(1, len(incidents) + 1)
    return incidents

## 4. Development result and optional one-time holdout

In [ ]:
truth_folder = "holdout_sealed" if RUN_HOLDOUT else "development"
truth_root = EVALUATION_ROOT / truth_folder
events = pd.read_parquet(truth_root / "fault_events.parquet")
intervals = pd.read_parquet(truth_root / "fault_entity_intervals.parquet")
if RUN_HOLDOUT:
    scores, alerts = score_partition(PARTITION)
    exposure = partition_exposure(
        scores, policy["primary_exposure_unit"],
        decisions["base_cadence_seconds"],
    )
else:
    alerts = pd.read_parquet(MODEL_ROOT / "development_alerts.parquet")
    exposure = configuration["development_exposure_value"]
    if not alerts.empty:
        alerts["score_percentile"] = score_percentiles(
            bundle, configuration["model_id"], alerts["peak_score"]
        )
    else:
        alerts["score_percentile"] = pd.Series(dtype=float)
    if configuration["threshold"] == 0:
        raise ValueError("Exceedance ratio requires a non-zero threshold")
    alerts["exceedance_ratio"] = (
        alerts["peak_score"] / configuration["threshold"]
    )
result = evaluate_alerts(
    alerts, events, intervals,
    exposure_value=exposure,
    exposure_unit=policy["primary_exposure_unit"],
    decision_horizon_seconds=policy["decision_horizon_seconds"],
    grouping_window_seconds=policy["grouping_window_seconds"],
    entity_groups=entity_groups,
)
ranked_incidents = rank_incidents(
    alerts, entity_groups, policy["grouping_window_seconds"]
)
ranked_incidents.insert(0, "partition", PARTITION)
evaluation_summary = result["metrics"].copy()
evaluation_summary.insert(0, "partition", PARTITION)
fault_type_results = result["fault_type_results"].copy()
fault_type_results.insert(0, "partition", PARTITION)
prediction_comparison = pd.DataFrame()
if RUN_HOLDOUT:
    observed_metrics = evaluation_summary.set_index("metric")["value"]
    prediction_rows = []
    for metric_name, prediction in holdout_prediction["predictions"].items():
        observed = float(observed_metrics.loc[metric_name])
        prediction_rows.append({
            "metric": metric_name,
            "expected": prediction["expected"],
            "confirmation_min": prediction["confirmation_min"],
            "confirmation_max": prediction["confirmation_max"],
            "observed": observed,
            "confirmed": (
                prediction["confirmation_min"] <= observed
                <= prediction["confirmation_max"]
            ),
        })
    prediction_comparison = pd.DataFrame(prediction_rows)

display(evaluation_summary.round(4))
display(fault_type_results)
if RUN_HOLDOUT:
    display(prediction_comparison)
display(ranked_incidents.head(25))

figure, axis = plt.subplots(figsize=(10, 4))
shown = ranked_incidents.head(20)
axis.barh(shown["incident_id"], shown["peak_exceedance_ratio"])
axis.invert_yaxis()
axis.set(xlabel="peak score / threshold", title="Highest-priority incidents")
figure.tight_layout(); plt.show()

## 5. Save the product-facing outputs

In [ ]:
with new_output_directory(FINAL_ROOT) as output:
    ranked_incidents.to_csv(output / "ranked_incidents.csv", index=False)
    evaluation_summary.to_csv(output / "evaluation_summary.csv", index=False)
    fault_type_results.to_csv(output / "fault_type_results.csv", index=False)
    model_fault_type_comparison.to_csv(
        output / "model_comparison_by_fault_type.csv", index=False
    )
    if RUN_HOLDOUT:
        prediction_comparison.to_csv(
            output / "holdout_prediction_comparison.csv", index=False
        )

print("Saved:", FINAL_ROOT)
print("Evaluated partition:", PARTITION)
if not RUN_HOLDOUT:
    print("Holdout remains sealed. Set RUN_HOLDOUT=1 only after the model is frozen.")
work.cleanup()